# naija-asr-eval: how well do open speech-recognition models hear Nigerian English?

This notebook runs the whole evaluation end to end on Google Colab's free GPU:

1. **Inspect** each dataset's metadata before filtering, so we can see the real field values.
2. **Sample** clips with a fixed seed:
   - **SLR70**: Google's crowdsourced Nigerian English corpus. ~300 clips, balanced female/male, spread across as many speakers as possible. **No approval needed.**
   - **AfriSpeech-200** (once Hugging Face approves your access): ~75 general-domain clips each for Yoruba, Igbo, Hausa and other Nigerian accents.
   - **LibriSpeech test-clean**: ~300 clips of standard (mostly American) read English as the baseline. Always on.
3. **Fetch audio** for the selected clips only, resampled to 16 kHz.
4. **Transcribe** with Whisper tiny, base and small, forced to English. Results are saved to CSV after every batch, so a disconnect loses at most one batch.
5. **Score** with `metrics.py`: corpus WER with 95% bootstrap CIs, substitutions/deletions/insertions, proper-noun accuracy, top substitutions and worst clips.
6. **Chart** WER by group and model.
7. *(Optional)* Score your own phone recordings the same way.

**Choosing datasets:** set `DATASETS` in the Configuration cell. It starts as `["slr70"]`. Once AfriSpeech access is approved, change it to `["slr70", "afrispeech"]` and run all cells again. Saved samples and transcriptions are reused, so only the new AfriSpeech clips are processed.

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- *For AfriSpeech only:*
  - Request access at [intronhealth/afrispeech-200](https://huggingface.co/datasets/intronhealth/afrispeech-200). Approval is manual.
  - Create a Hugging Face **read** token and add it in Colab Secrets (the key icon) as `HF_TOKEN`, with notebook access switched on.

**Privacy:** audio lives only in Colab's temporary disk and your private Google Drive (`MyDrive/naija-asr-eval/audio_cache`). The results folder holds only CSVs and a chart, and the last cell checks that before zipping. Never commit audio.

In [ ]:
# datasets 4.0 removed support for dataset loading scripts, which AfriSpeech-200 uses.
# Expected and harmless: pip may print "ERROR: ... gcsfs ... requires fsspec==..., but you have
# fsspec ... which is incompatible". Colab's preinstalled gcsfs wants a newer fsspec than
# datasets 3.x allows; this notebook never uses gcsfs. Carry on.
%pip install -q "datasets>=3.0,<4.0" librosa soundfile

## 0. Get the code

`metrics.py` (scoring) and `slr70.py` (SLR70 parsing and sampling) are plain Python files, unit-tested on a laptop. Here we either clone the GitHub repo or upload them.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = ""  # e.g. "https://github.com/<your-username>/naija-asr-eval.git" once it is pushed
CODE_FILES = ["metrics.py", "slr70.py"]

if not all(Path(f).exists() for f in CODE_FILES):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
        os.chdir("repo")
    else:
        from google.colab import files
        print(f"Upload {' and '.join(CODE_FILES)} (plus test_*.py files to run the tests here).")
        files.upload()

missing_code = [f for f in CODE_FILES if not Path(f).exists()]
if missing_code:
    raise FileNotFoundError(f"Missing {missing_code}: set REPO_URL or upload them, then rerun this cell.")
print("Code found in", Path.cwd())

In [ ]:
# Optional: run the same unit tests that pass on the laptop, to prove the code is unchanged here.
test_files = [f for f in ["test_metrics.py", "test_slr70.py"] if Path(f).exists()]
if test_files:
    subprocess.run(["python", "-m", "pytest", "-q", *test_files], check=False)

## Configuration

Every number that shapes the experiment is set here, so the method can be read in one place.

In [ ]:
import csv
import io
import json
import re
import shutil
import subprocess
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path, PurePosixPath

import datasets
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers
from datasets import Audio, get_dataset_config_names, load_dataset
from huggingface_hub import hf_hub_download, login
from matplotlib.ticker import PercentFormatter
from transformers import WhisperForConditionalGeneration, WhisperProcessor

import metrics
import slr70

try:
    from google.colab import drive, files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---- Which Nigerian datasets to run. LibriSpeech (the baseline) is always included.
DATASETS = ["slr70"]               # add "afrispeech" once Hugging Face approves your access
KNOWN_DATASETS = {"slr70", "afrispeech"}
if not DATASETS or set(DATASETS) - KNOWN_DATASETS:
    raise ValueError(f"DATASETS must be a non-empty subset of {sorted(KNOWN_DATASETS)}, got {DATASETS}")

SEED = 42
MAX_DURATION_S = 30.0              # Whisper hears 30-second windows; longer clips need chunking
SAMPLE_RATE = 16_000
FORCE_RESAMPLE = False             # True ignores saved manifests and samples again

# ---- SLR70: Google's crowdsourced Nigerian English (OpenSLR 70), CC BY-SA 4.0
SLR70_CLIPS = 300                  # half female, half male
SLR70_GROUP = "Nigerian English (SLR70)"
SLR70_MIRRORS = [
    "https://openslr.trmal.net/resources/70/",
    "https://openslr.elda.org/resources/70/",
    "https://openslr.magicdatatech.com/resources/70/",
    "https://www.openslr.org/resources/70/",
]
SLR70_FILES = {"female": ("line_index_female.tsv", "en_ng_female.zip"),
               "male": ("line_index_male.tsv", "en_ng_male.zip")}

# ---- AfriSpeech-200 (gated), CC BY-NC-SA 4.0
AFRISPEECH_ID = "intronhealth/afrispeech-200"
AFRISPEECH_SPLITS = ["test", "dev"]   # held-out splits only
CLIPS_PER_GROUP = 75
MAX_CLIPS_PER_SPEAKER = 3          # stops a few talkative speakers dominating a group
NIGERIA = "NG"
DOMAIN = "general"
MAIN_ACCENTS = {"yoruba": "Yoruba", "igbo": "Igbo", "hausa": "Hausa"}
OTHER_GROUP = "Other Nigerian"
AFRISPEECH_GROUPS = ["Yoruba", "Igbo", "Hausa", OTHER_GROUP]
EXCLUDED_ACCENTS = {"english"}     # an "english" accent label says nothing about a Nigerian language background
POOLED_GROUP = "All AfriSpeech"    # pooled within AfriSpeech only: pooling across corpora mixes recording conditions

# ---- LibriSpeech baseline, CC BY 4.0
LIBRISPEECH_ID = "openslr/librispeech_asr"
LIBRISPEECH_CLIPS = 300
LIBRISPEECH_MAX_PER_SPEAKER = 8    # test-clean has only 40 speakers, so 3 each would give 120 clips
BASELINE_GROUP = "LibriSpeech"

NIGERIAN_GROUPS = ((AFRISPEECH_GROUPS if "afrispeech" in DATASETS else [])
                   + ([SLR70_GROUP] if "slr70" in DATASETS else []))
ALL_GROUPS = NIGERIAN_GROUPS + [BASELINE_GROUP]
ACTIVE_DATASETS = [d for d in ["afrispeech", "slr70"] if d in DATASETS] + ["librispeech"]

MODELS = ["openai/whisper-tiny", "openai/whisper-base", "openai/whisper-small"]
BATCH_SIZE = 16

if IN_COLAB:
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/naija-asr-eval")
    SLR70_DIR = Path("/content/slr70")         # Colab's local disk: fast, private, wiped on disconnect
else:
    WORK_DIR = Path("work")
    SLR70_DIR = WORK_DIR / "slr70_download"
AUDIO_CACHE = WORK_DIR / "audio_cache"          # private; never copied into the repo
RESULTS_DIR = WORK_DIR / "results"              # CSVs and charts only; this is what you commit
PRED_DIR = RESULTS_DIR / "predictions"
for folder in (AUDIO_CACHE, PRED_DIR, SLR70_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def manifest_path(dataset: str) -> Path:
    return RESULTS_DIR / f"manifest_{dataset}.csv"


print("Datasets:", ACTIVE_DATASETS, "| groups:", ALL_GROUPS)
print(f"datasets {datasets.__version__} | transformers {transformers.__version__} | torch {torch.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")

## Hugging Face login

AfriSpeech-200 is gated, so it needs your token, which is read from Colab Secrets and never appears in the notebook. SLR70 (from OpenSLR) and LibriSpeech need no approval, so the token is optional when AfriSpeech is off.

In [ ]:
def read_hf_token() -> str | None:
    if not IN_COLAB:
        return None
    try:
        return userdata.get("HF_TOKEN")
    except Exception:  # userdata raises its own SecretNotFoundError / NotebookAccessError types
        return None


hf_token = read_hf_token()
if hf_token:
    login(token=hf_token)
elif not IN_COLAB:
    print("Not on Colab: using the token saved by `huggingface-cli login`, if any.")
elif "afrispeech" in DATASETS:
    raise RuntimeError("AfriSpeech needs HF_TOKEN: add it under the key icon (Secrets) with notebook access on.")
else:
    print("No HF_TOKEN found: fine, because SLR70 and LibriSpeech need no approval.")

## 1. Inspect the metadata before filtering

Each dataset ships small metadata files next to its audio. We read those first (a few hundred KB) to check the real values before any filtering or large download.

### 1a. SLR70

Two TSV files, one per gender. Each line is `file ID <TAB> transcript`, and the file ID is `<ngf|ngm>_<speaker number>_<utterance number>`, so gender and speaker come from the ID. Check in the output:
- how many speakers each gender has, and how unevenly they recorded (this is why we sample round-robin across speakers)
- whether any lines were skipped as malformed
- that transcripts are mixed case, which the proper-noun heuristic needs

In [ ]:
def download_file(filename: str, folder: Path, mirrors: list[str] = SLR70_MIRRORS) -> Path:
    """Download from the first mirror that works. Skips files already downloaded.

    The download goes to '<name>.part' and is renamed only when complete, so a
    file cut off by a disconnect is never mistaken for a finished one.
    """
    target = folder / filename
    if target.exists() and (target.suffix != ".zip" or zipfile.is_zipfile(target)):
        return target
    partial = target.with_name(target.name + ".part")
    for base in mirrors:
        url = base + filename
        print(f"  downloading {url}")
        started = time.time()
        try:
            with urllib.request.urlopen(url, timeout=60) as response, partial.open("wb") as out:
                shutil.copyfileobj(response, out, length=1 << 20)
        except (urllib.error.URLError, TimeoutError, OSError) as error:
            print(f"    failed: {error}")
            continue
        if target.suffix == ".zip" and not zipfile.is_zipfile(partial):
            print("    not a valid zip, trying the next mirror")
            continue
        partial.replace(target)
        print(f"    {target.stat().st_size / 1e6:.0f} MB in {time.time() - started:.0f} s")
        return target
    raise RuntimeError(f"Could not download {filename} from any mirror")


def load_slr70_index() -> list[slr70.Utterance]:
    utterances = []
    for gender, (tsv_name, _) in SLR70_FILES.items():
        text = download_file(tsv_name, SLR70_DIR).read_text(encoding="utf-8")
        parsed, problems = slr70.parse_line_index(text)
        print(f"{tsv_name}: {len(parsed):,} utterances, {len(problems)} malformed lines skipped")
        for problem in problems[:5]:
            print("   ", problem)
        mislabelled = [u.file_id for u in parsed if u.gender != gender]
        if mislabelled:
            print(f"    warning: {len(mislabelled)} IDs in the {gender} file have the other gender's prefix")
        utterances += parsed
    return utterances


def describe_slr70(utterances: list[slr70.Utterance]) -> None:
    table = pd.DataFrame([vars(u) for u in utterances])
    per_speaker = table.groupby(["gender", "speaker"]).size()
    print("\nClips per speaker, by gender:")
    print(per_speaker.groupby("gender").describe()[["count", "min", "50%", "max"]]
          .rename(columns={"count": "speakers", "50%": "median"}).to_string())
    print("\nFive random transcripts:")
    for text in table["transcript"].sample(5, random_state=SEED):
        print("  ", text)
    print(f"\nShare containing lowercase letters: {table['transcript'].str.contains(r'[a-z]').mean():.1%}")


if "slr70" in DATASETS:
    slr70_index = load_slr70_index()
    describe_slr70(slr70_index)
else:
    print("SLR70 is off (not in DATASETS).")

### 1b. AfriSpeech-200 (only when `"afrispeech"` is in `DATASETS`)

The metadata is in `transcripts/<split>.csv` on Hugging Face. Things to check:
- the exact spelling of the `country` and `domain` values (the filter expects `NG` and `general`)
- the accent labels, including **mixed labels** such as `yoruba, hausa`, which we exclude
- whether Yoruba/Igbo/Hausa speakers also appear with other countries

In [ ]:
REQUIRED_COLUMNS = ["user_ids", "accent", "country", "domain", "duration", "transcript", "audio_paths"]


def load_afrispeech_metadata(splits: list[str]) -> pd.DataFrame:
    """Download the metadata CSV for each split and stack them, tagging each row's split."""
    frames = []
    for split in splits:
        path = hf_hub_download(AFRISPEECH_ID, f"transcripts/{split}.csv", repo_type="dataset")
        frame = pd.read_csv(path, keep_default_na=False)
        frame["source_split"] = split
        frames.append(frame)
    meta = pd.concat(frames, ignore_index=True)

    missing = [c for c in REQUIRED_COLUMNS if c not in meta.columns]
    if missing:
        raise KeyError(f"Metadata is missing columns {missing}. Columns found: {list(meta.columns)}")
    return meta


def clean_metadata(meta: pd.DataFrame) -> pd.DataFrame:
    """Normalise case and whitespace so filters compare like with like."""
    cleaned = meta.copy()
    cleaned["accent"] = cleaned["accent"].astype(str).str.strip().str.lower()
    cleaned["country"] = cleaned["country"].astype(str).str.strip().str.upper()
    cleaned["domain"] = cleaned["domain"].astype(str).str.strip().str.lower()
    cleaned["transcript"] = cleaned["transcript"].astype(str).str.strip()
    cleaned["duration"] = pd.to_numeric(cleaned["duration"], errors="coerce")
    if "gender" in cleaned.columns:
        cleaned["gender"] = cleaned["gender"].astype(str).str.strip().str.lower()
    return cleaned


def describe_afrispeech(raw: pd.DataFrame, meta: pd.DataFrame) -> None:
    print(f"{len(raw):,} rows from splits {AFRISPEECH_SPLITS}")
    print("Columns:", list(raw.columns))
    for column in ["source_split", "domain", "country"]:
        print(f"\n--- raw values of '{column}' ---")
        print(raw[column].value_counts(dropna=False).to_string())

    ng_general = meta[(meta["country"] == NIGERIA) & (meta["domain"] == DOMAIN)]
    print(f"\n--- accent labels among {len(ng_general):,} Nigerian general-domain clips ---")
    print(ng_general["accent"].value_counts().to_string())

    print("\n--- main accents: country x domain (all rows, before filtering) ---")
    main_rows = meta[meta["accent"].isin(MAIN_ACCENTS)]
    print(pd.crosstab([main_rows["accent"], main_rows["country"]], main_rows["domain"]).to_string())

    print("\n--- five random Nigerian general-domain references (the name heuristic needs mixed case) ---")
    for text in ng_general["transcript"].sample(5, random_state=SEED):
        print("  ", text)
    print(f"\nShare containing lowercase letters: {ng_general['transcript'].str.contains(r'[a-z]').mean():.1%}")


if "afrispeech" in DATASETS:
    afri_raw = load_afrispeech_metadata(AFRISPEECH_SPLITS)
    afri_meta = clean_metadata(afri_raw)
    describe_afrispeech(afri_raw, afri_meta)
else:
    print("AfriSpeech is off. Add \"afrispeech\" to DATASETS once your access is approved.")

## 2. Filter and sample

**Read the output above before running this.** If the values differ from what the Configuration constants expect, fix the constants first.

Each dataset's sample is saved as `results/manifest_<dataset>.csv`. On a rerun a saved manifest is reused, so turning AfriSpeech on later does not change the SLR70 or LibriSpeech samples. Every manifest has the same columns: `clip_id, dataset, group, accent, gender, source, source_split, audio_key, speaker, duration_s, reference`. Speaker IDs are replaced with short per-run labels.

### SLR70 rules
- Duration ≤ 30 s, read from each WAV header inside the zip without decoding the audio. This step downloads the two zips (~1.2 GB) to Colab's temporary disk, never to your laptop.
- `slr70.balanced_sample`: 150 female + 150 male. Within each gender we go **round-robin across speakers**: every speaker gives one clip before anyone gives a second. SLR70 has few speakers who each recorded many sentences, so a fixed per-speaker cap like AfriSpeech's would not reach 300. Round-robin gets the widest spread of voices for whatever n we ask for.

In [ ]:
MANIFEST_COLUMNS = ["clip_id", "dataset", "group", "accent", "gender", "source", "source_split",
                          "audio_key", "speaker", "duration_s", "reference"]


def anonymise_speakers(ids: pd.Series, prefix: str) -> pd.Series:
    """Replace dataset speaker IDs with short per-run labels (spk001, ...) for the committed CSVs."""
    codes = {speaker: f"{prefix}{i:03d}" for i, speaker in enumerate(ids.unique(), start=1)}
    return ids.map(codes)


def zip_members(archive: zipfile.ZipFile) -> dict[str, str]:
    """Map 'ngm_02436_01770933554' -> its path inside the zip, wherever the folder layout puts it."""
    return {PurePosixPath(name).stem: name for name in archive.namelist() if name.lower().endswith(".wav")}


def slr70_durations(utterances: list[slr70.Utterance]) -> dict[str, float]:
    """Duration of every utterance that has audio, read from WAV headers only."""
    durations = {}
    for gender, (_, zip_name) in SLR70_FILES.items():
        with zipfile.ZipFile(download_file(zip_name, SLR70_DIR)) as archive:
            members = zip_members(archive)
            for utterance in utterances:
                member = members.get(utterance.file_id) if utterance.gender == gender else None
                if member is None:
                    continue
                try:
                    with archive.open(member) as handle:
                        durations[utterance.file_id] = sf.info(handle).duration
                except Exception as error:  # one unreadable file must not stop the run
                    print(f"  cannot read {member}: {error}")
    return durations


def build_slr70_manifest() -> pd.DataFrame:
    durations = slr70_durations(slr70_index)
    no_audio = [u for u in slr70_index if u.file_id not in durations]
    too_long = [u for u in slr70_index if durations.get(u.file_id, 0) > MAX_DURATION_S]
    eligible = [u for u in slr70_index if durations.get(u.file_id, np.inf) <= MAX_DURATION_S]
    print(f"  removed {len(no_audio):>5}  transcript with no readable audio in the zip")
    print(f"  removed {len(too_long):>5}  longer than {MAX_DURATION_S:.0f} s")
    print(f"  {len(eligible):,} eligible clips")

    chosen = slr70.balanced_sample(eligible, SLR70_CLIPS, SEED)
    picked = pd.DataFrame([vars(u) for u in chosen])
    for gender, rows in picked.groupby("gender"):
        target = SLR70_CLIPS // 2 + (SLR70_CLIPS % 2 if gender == "female" else 0)
        short = f"  <-- SHORT by {target - len(rows)}" if len(rows) < target else ""
        per_speaker = rows["speaker"].value_counts()
        print(f"{SLR70_GROUP} {gender:<6} {len(rows):>3} clips from {len(per_speaker):>3} speakers "
              f"(max {per_speaker.max()} per speaker){short}")

    zip_by_gender = {g: zip_name for g, (_, zip_name) in SLR70_FILES.items()}
    return pd.DataFrame({
        "clip_id": "slr70-" + picked["file_id"],
        "dataset": "slr70",
        "group": SLR70_GROUP,
        "accent": "",
        "gender": picked["gender"],
        "source": picked["gender"].map(zip_by_gender),
        "source_split": "all",
        "audio_key": picked["file_id"],
        "speaker": anonymise_speakers(picked["speaker"], "slr70-spk"),
        "duration_s": picked["file_id"].map(durations).round(2),
        "reference": picked["transcript"],
    })[MANIFEST_COLUMNS]

### AfriSpeech rules (only when enabled)
- `country == NG`, `domain == general`, duration ≤ 30 s, non-empty transcript.
- Groups: `yoruba`, `igbo` and `hausa` exactly. **Other Nigerian** is every other Nigerian accent, minus mixed labels that mention a main accent (e.g. `igbo and yoruba`) and the label `english`.
- Each clip must belong to a per-accent dataset config, so its audio can be streamed without touching other accents.
- Shuffle with a fixed seed, keep at most `MAX_CLIPS_PER_SPEAKER` clips per speaker, then take the first 75. A group with fewer eligible clips keeps all of them, and **the shortfall is reported**. Expect this for Hausa, which has ~51 eligible clips.

In [ ]:
def accent_to_config(accent: str) -> str:
    """'yala mbembe' -> 'yala-mbembe', 'ijaw(nembe)' -> 'ijaw-nembe' (the dataset's config naming)."""
    return re.sub(r"[^a-z]+", "-", accent).strip("-")


def assign_group(accent: str) -> str | None:
    if accent in MAIN_ACCENTS:
        return MAIN_ACCENTS[accent]
    if accent in EXCLUDED_ACCENTS:
        return None
    if any(re.search(rf"\b{main}\b", accent) for main in MAIN_ACCENTS):
        return None  # mixed label such as "yoruba, hausa": it belongs to no single group
    return OTHER_GROUP


def eligible_afrispeech(meta: pd.DataFrame, available_configs: set[str]) -> pd.DataFrame:
    """Apply the filters one at a time, printing how many clips each rule removes."""
    rules = [
        ("country is not NG", meta["country"] != NIGERIA),
        ("domain is not general", meta["domain"] != DOMAIN),
        ("empty transcript", ~meta["transcript"].str.contains(r"\w")),
        (f"longer than {MAX_DURATION_S:.0f} s or unknown duration", ~(meta["duration"] <= MAX_DURATION_S)),
    ]
    keep = pd.Series(True, index=meta.index)
    for reason, drop in rules:
        print(f"  removed {int((keep & drop).sum()):>6,}  {reason}")
        keep &= ~drop

    eligible = meta[keep].copy()
    eligible["group"] = eligible["accent"].map(assign_group)
    eligible["hf_config"] = eligible["accent"].map(accent_to_config)
    no_group = eligible["group"].isna()
    no_config = ~eligible["hf_config"].isin(available_configs)
    print(f"  removed {int(no_group.sum()):>6,}  mixed or excluded accent label")
    print(f"  removed {int((~no_group & no_config).sum()):>6,}  accent has no dataset config: "
          f"{sorted(eligible.loc[~no_group & no_config, 'accent'].unique())}")
    eligible = eligible[~no_group & ~no_config]
    eligible["audio_key"] = eligible["audio_paths"].map(lambda p: PurePosixPath(p).stem)
    print(f"  {len(eligible):,} eligible clips")
    return eligible


def sample_capped(frame: pd.DataFrame, n: int, speaker_col: str, max_per_speaker: int, seed: int) -> pd.DataFrame:
    """Seeded random sample with at most max_per_speaker clips from any one speaker."""
    shuffled = frame.sample(frac=1, random_state=seed)
    capped = shuffled.groupby(speaker_col, sort=False).head(max_per_speaker)
    return capped.head(n)


def build_afrispeech_manifest() -> pd.DataFrame:
    available_configs = set(get_dataset_config_names(AFRISPEECH_ID, trust_remote_code=True))
    eligible = eligible_afrispeech(afri_meta, available_configs)

    picked = []
    for group in AFRISPEECH_GROUPS:
        pool = eligible[eligible["group"] == group]
        chosen = sample_capped(pool, CLIPS_PER_GROUP, "user_ids", MAX_CLIPS_PER_SPEAKER, SEED)
        shortfall = CLIPS_PER_GROUP - len(chosen)
        note = f"  <-- SHORT by {shortfall}: not enough eligible clips/speakers" if shortfall > 0 else ""
        print(f"{group:<15} {len(chosen):>3} clips from {chosen['user_ids'].nunique():>3} speakers "
              f"(pool {len(pool):,} clips){note}")
        picked.append(chosen)
    chosen = pd.concat(picked, ignore_index=True)

    other = chosen[chosen["group"] == OTHER_GROUP]
    print("\nAccent mix in Other Nigerian:", other["accent"].value_counts().to_dict())

    return pd.DataFrame({
        "clip_id": "afri-" + chosen["audio_key"],
        "dataset": "afrispeech",
        "group": chosen["group"],
        "accent": chosen["accent"],
        "gender": chosen["gender"] if "gender" in chosen.columns else "",
        "source": chosen["hf_config"],
        "source_split": chosen["source_split"],
        "audio_key": chosen["audio_key"],
        "speaker": anonymise_speakers(chosen["user_ids"], "afri-spk"),
        "duration_s": chosen["duration"].round(2),
        "reference": chosen["transcript"],
    })[MANIFEST_COLUMNS]

### LibriSpeech baseline (always on)

LibriSpeech test-clean is the standard "clean, read, mostly American English" benchmark. We stream it once to read IDs, speakers and durations (from the FLAC header, without decoding audio), then sample 300 clips with the same seed.

LibriSpeech references are ALL CAPS with no punctuation, so the proper-noun heuristic cannot run on them. `metrics.py` returns "not applicable" for these clips instead of a wrong number.

In [ ]:
def open_librispeech_stream():
    stream = load_dataset(LIBRISPEECH_ID, "clean", split="test", streaming=True, trust_remote_code=True)
    return stream.cast_column("audio", Audio(decode=False))


def build_librispeech_manifest() -> pd.DataFrame:
    rows = []
    for example in open_librispeech_stream():
        duration = sf.info(io.BytesIO(example["audio"]["bytes"])).duration
        rows.append({"libri_id": example["id"], "speaker_id": example["speaker_id"],
                     "duration": duration, "text": example["text"]})
    pool = pd.DataFrame(rows)
    pool = pool[pool["duration"] <= MAX_DURATION_S]
    chosen = sample_capped(pool, LIBRISPEECH_CLIPS, "speaker_id", LIBRISPEECH_MAX_PER_SPEAKER, SEED)
    print(f"LibriSpeech     {len(chosen):>3} clips from {chosen['speaker_id'].nunique():>3} speakers (pool {len(pool):,})")
    return pd.DataFrame({
        "clip_id": "libri-" + chosen["libri_id"],
        "dataset": "librispeech",
        "group": BASELINE_GROUP,
        "accent": "",
        "gender": "",
        "source": "clean",
        "source_split": "test-clean",
        "audio_key": chosen["libri_id"],
        "speaker": anonymise_speakers(chosen["speaker_id"], "libri-spk"),
        "duration_s": chosen["duration"].round(2),
        "reference": chosen["text"],
    })[MANIFEST_COLUMNS]

In [ ]:
MANIFEST_BUILDERS = {"slr70": build_slr70_manifest, "afrispeech": build_afrispeech_manifest,
                     "librispeech": build_librispeech_manifest}
TEXT_COLUMNS = {c: str for c in ["clip_id", "audio_key", "reference", "speaker", "source", "gender", "accent"]}


def save_manifests(manifest: pd.DataFrame) -> None:
    for dataset, rows in manifest.groupby("dataset"):
        rows.to_csv(manifest_path(dataset), index=False)


parts = []
for dataset in ACTIVE_DATASETS:
    path = manifest_path(dataset)
    if path.exists() and not FORCE_RESAMPLE:
        # dtype=str stops pandas reading a file hash such as "12e45..." as a number.
        part = pd.read_csv(path, keep_default_na=False, dtype=TEXT_COLUMNS)
        print(f"{dataset}: loaded the saved sample ({len(part)} clips)")
    else:
        print(f"{dataset}: sampling")
        part = MANIFEST_BUILDERS[dataset]()
        part.to_csv(path, index=False)
    parts.append(part)

manifest = pd.concat(parts, ignore_index=True)
if manifest["clip_id"].duplicated().any():
    raise ValueError("Duplicate clip IDs in the manifest")
print(manifest.groupby(["dataset", "group"]).agg(
    clips=("clip_id", "size"), speakers=("speaker", "nunique"),
    minutes=("duration_s", lambda s: round(s.sum() / 60, 1))))

## 3. Fetch audio for the selected clips only

Each selected clip is decoded, mixed to mono, resampled to 16 kHz and cached as FLAC in your private Drive. A rerun skips clips that are already cached, so after a disconnect nothing is downloaded again unless a clip is missing.

- **SLR70:** we read only the selected WAV files out of the zips (`archive.read(member)`), never extracting the rest.
- **AfriSpeech:** audio comes as one archive per accent and split, so we stream only the configs our clips come from. Decoding is off while streaming (`Audio(decode=False)`), and a stream stops once all its clips are found. Clips are matched by the audio file name: the metadata's `audio_ids` column is a different hash.
- **LibriSpeech:** streamed once more, keeping only the selected IDs.

In [ ]:
SPLIT_ALIASES = {"dev": ["dev", "validation"], "test": ["test"]}


def cache_path(clip_id: str) -> Path:
    return AUDIO_CACHE / f"{clip_id}.flac"


def decode_to_16k(audio_bytes: bytes) -> np.ndarray:
    data, rate = sf.read(io.BytesIO(audio_bytes), dtype="float32", always_2d=True)
    mono = data.mean(axis=1)
    if rate != SAMPLE_RATE:
        mono = librosa.resample(mono, orig_sr=rate, target_sr=SAMPLE_RATE)
    return mono.astype(np.float32)


def save_audio(clip_id: str, audio: np.ndarray) -> None:
    sf.write(cache_path(clip_id), np.clip(audio, -1.0, 1.0), SAMPLE_RATE, format="FLAC", subtype="PCM_16")


def load_audio(clip_id: str) -> np.ndarray:
    audio, rate = sf.read(cache_path(clip_id), dtype="float32")
    if rate != SAMPLE_RATE:
        raise ValueError(f"{clip_id} is cached at {rate} Hz, expected {SAMPLE_RATE}")
    return audio


def open_afrispeech_stream(config: str, split: str):
    """Stream one accent/split with audio decoding off. Tries 'dev' and 'validation' for the dev split."""
    last_error = None
    for name in SPLIT_ALIASES.get(split, [split]):
        try:
            stream = load_dataset(AFRISPEECH_ID, config, split=name, streaming=True, trust_remote_code=True)
            return stream.cast_column("audio", Audio(decode=False))
        except ValueError as error:  # raised for an unknown split name
            last_error = error
    raise last_error


def afrispeech_key(example: dict) -> str:
    for field in ("path", "audio_id"):
        if example.get(field):
            return PurePosixPath(str(example[field])).stem
    return PurePosixPath(example["audio"]["path"]).stem


def fetch_from_stream(stream, wanted: dict[str, str], key_fn) -> list[str]:
    """Cache audio for every stream row whose key is in `wanted` (key -> clip_id).

    Stops as soon as everything is found. Returns clip IDs that failed to decode;
    clips never seen are left in `wanted`.
    """
    failed = []
    for example in stream:
        clip_id = wanted.pop(key_fn(example), None)
        if clip_id is None:
            continue
        try:
            audio_bytes = example["audio"]["bytes"]
            if audio_bytes is None:
                with open(example["audio"]["path"], "rb") as handle:
                    audio_bytes = handle.read()
            save_audio(clip_id, decode_to_16k(audio_bytes))
        except Exception as error:  # one corrupt file must not stop the whole run
            print(f"    could not decode {clip_id}: {error}")
            failed.append(clip_id)
        if not wanted:
            break
    return failed


def fetch_from_zip(zip_path: Path, wanted: dict[str, str]) -> list[str]:
    """Same contract as fetch_from_stream, reading only the wanted members of a zip."""
    failed = []
    with zipfile.ZipFile(zip_path) as archive:
        members = zip_members(archive)
        for key, clip_id in list(wanted.items()):
            member = members.get(key)
            if member is None:
                continue  # stays in `wanted`, so it is reported as not found
            del wanted[key]
            try:
                save_audio(clip_id, decode_to_16k(archive.read(member)))
            except Exception as error:
                print(f"    could not decode {clip_id}: {error}")
                failed.append(clip_id)
    return failed


def fetch_all_audio(manifest: pd.DataFrame) -> list[str]:
    todo = manifest[~manifest["clip_id"].map(lambda c: cache_path(c).exists())]
    print(f"{len(manifest) - len(todo)} clips already cached, {len(todo)} to fetch")
    problems = []
    for (dataset, source, split), rows in todo.groupby(["dataset", "source", "source_split"]):
        wanted = dict(zip(rows["audio_key"], rows["clip_id"]))
        print(f"  {dataset}/{source}/{split}: {len(wanted)} clips")
        started = time.time()
        if dataset == "slr70":
            problems += fetch_from_zip(download_file(source, SLR70_DIR), wanted)
        elif dataset == "afrispeech":
            problems += fetch_from_stream(open_afrispeech_stream(source, split), wanted, afrispeech_key)
        else:
            problems += fetch_from_stream(open_librispeech_stream(), wanted, lambda example: example["id"])
        if wanted:
            print(f"    {len(wanted)} clips were not found")
            problems += list(wanted.values())
        print(f"    done in {time.time() - started:.0f} s")
    return problems

In [ ]:
problems = fetch_all_audio(manifest)
if problems:
    print(f"\n{len(problems)} clips have no audio and are dropped from the manifests: {problems[:10]}")
    manifest = manifest[~manifest["clip_id"].isin(problems)].reset_index(drop=True)
    save_manifests(manifest)
print(f"{len(manifest)} clips ready")

## 4. Transcribe with Whisper

Settings shared by all three models:
- **`language="en", task="transcribe"`** forces English transcription. Without it Whisper may detect another language for accented speech, or translate.
- Greedy decoding (the default), fp16 on the GPU, batches of 16.
- After every batch the results are appended to `results/predictions/<model>.csv`. On a rerun, clips already in that file are skipped, so a Colab disconnect costs at most one batch, and turning on a new dataset only transcribes its new clips.

In [ ]:
PREDICTION_FIELDS = ["clip_id", "model", "hypothesis"]


def short_name(model_name: str) -> str:
    return model_name.split("/")[-1]


def completed_clip_ids(path: Path) -> set[str]:
    if not path.exists():
        return set()
    return set(pd.read_csv(path, usecols=["clip_id"], keep_default_na=False, dtype=str)["clip_id"])


def append_rows(path: Path, rows: list[dict]) -> None:
    is_new = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=PREDICTION_FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerows(rows)
        handle.flush()


def transcribe_all(model_name: str, clip_ids: list[str], out_path: Path) -> None:
    done = completed_clip_ids(out_path)
    todo = [c for c in clip_ids if c not in done]
    print(f"{short_name(model_name)}: {len(clip_ids) - len(todo)} done, {len(todo)} to go")
    if not todo:
        return

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name, torch_dtype=dtype).to(device).eval()

    started = time.time()
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start:start + BATCH_SIZE]
        features = processor([load_audio(c) for c in batch], sampling_rate=SAMPLE_RATE,
                             return_tensors="pt").input_features.to(device, dtype)
        with torch.inference_mode():
            token_ids = model.generate(features, language="en", task="transcribe")
        texts = processor.batch_decode(token_ids, skip_special_tokens=True)
        append_rows(out_path, [{"clip_id": c, "model": model_name, "hypothesis": t.strip()}
                               for c, t in zip(batch, texts)])
        finished = start + len(batch)
        print(f"  {finished}/{len(todo)} clips, {time.time() - started:.0f} s", end="\r")
    print()

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

In [ ]:
clip_ids = manifest["clip_id"].tolist()
for model_name in MODELS:
    transcribe_all(model_name, clip_ids, PRED_DIR / f"{short_name(model_name)}.csv")

## 5. Score

Each clip is scored with `metrics.score_clip`, using exactly the same rules for every dataset. Group numbers are **corpus-level**: total errors divided by total reference words, so long clips weigh more than short ones. The 95% confidence intervals come from 2,000 bootstrap resamples of whole clips.

**Gap** = WER of a Nigerian group minus WER of LibriSpeech, for the same model, with its own bootstrap CI. If a bigger model closes the gap (research question 2), the gap should shrink from tiny to small and its CI should move toward 0.

**All AfriSpeech** pools the four AfriSpeech groups. SLR70 is never pooled with AfriSpeech: the two corpora differ in recording setup, sentence sources and speaker selection, so a pooled number would mix those effects with accent.

The **proper-noun heuristic** counts capitalised words that do not start a sentence (and are not "I" or titles) as likely names. It is a heuristic, not named-entity recognition. It misses names at the start of a sentence and counts some capitalised non-names.

In [ ]:
DATASET_OF_GROUP = {**{g: "AfriSpeech" for g in AFRISPEECH_GROUPS + [POOLED_GROUP]},
                    SLR70_GROUP: "SLR70", BASELINE_GROUP: "LibriSpeech"}


def load_predictions(path: Path) -> pd.DataFrame:
    preds = pd.read_csv(path, keep_default_na=False, dtype=str)
    return preds.drop_duplicates("clip_id", keep="last")


def score_predictions(manifest: pd.DataFrame, preds: pd.DataFrame, model_name: str):
    """Score every clip that has a prediction. Returns (per-clip table, {clip_id: ClipScore})."""
    merged = manifest.merge(preds[["clip_id", "hypothesis"]], on="clip_id", how="inner")
    missing = len(manifest) - len(merged)
    if missing:
        print(f"  warning: {short_name(model_name)} has no prediction for {missing} clips")

    records, scores = [], {}
    for row in merged.itertuples(index=False):
        score = metrics.score_clip(row.reference, row.hypothesis)
        scores[row.clip_id] = score
        counts, names = score.counts, score.proper_nouns
        records.append({
            "model": short_name(model_name), "clip_id": row.clip_id, "group": row.group,
            "ref_words": counts.ref_words, "hits": counts.hits, "substitutions": counts.substitutions,
            "deletions": counts.deletions, "insertions": counts.insertions, "errors": counts.errors,
            "wer": counts.wer,
            "names_total": None if names is None else len(names),
            "names_correct": None if names is None else sum(n.correct for n in names),
            "reference": row.reference, "hypothesis": row.hypothesis,
            "alignment": metrics.format_alignment(score.alignment),
        })
    return pd.DataFrame(records), scores


def group_views(clips: pd.DataFrame, groups: list[str]) -> list[tuple[str, pd.DataFrame]]:
    views = [(g, clips[clips["group"] == g]) for g in groups]
    afri_present = [g for g in AFRISPEECH_GROUPS if g in groups and (clips["group"] == g).any()]
    if len(afri_present) >= 2:
        views.append((POOLED_GROUP, clips[clips["group"].isin(afri_present)]))
    return [(g, v) for g, v in views if not v.empty]


def summarise(clips: pd.DataFrame, groups: list[str]) -> pd.DataFrame:
    rows = []
    for model, model_clips in clips.groupby("model", sort=False):
        for group, view in group_views(model_clips, groups):
            wer = metrics.bootstrap_wer(view["errors"], view["ref_words"], seed=SEED)
            words = view["ref_words"].sum()
            row = {
                "model": model, "group": group, "clips": len(view), "ref_words": int(words),
                "wer": wer.value, "ci_low": wer.ci_low, "ci_high": wer.ci_high,
                "sub_rate": view["substitutions"].sum() / words,
                "del_rate": view["deletions"].sum() / words,
                "ins_rate": view["insertions"].sum() / words,
                "names_total": 0, "name_error_rate": np.nan, "name_ci_low": np.nan, "name_ci_high": np.nan,
            }
            named = view.dropna(subset=["names_total"])
            if named["names_total"].sum() > 0:
                wrong = named["names_total"] - named["names_correct"]
                names = metrics.bootstrap_ratio(wrong, named["names_total"], seed=SEED)
                row.update(names_total=int(named["names_total"].sum()), name_error_rate=names.value,
                           name_ci_low=names.ci_low, name_ci_high=names.ci_high)
            rows.append(row)
    return pd.DataFrame(rows)


def gaps_vs_baseline(clips: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model, model_clips in clips.groupby("model", sort=False):
        base = model_clips[model_clips["group"] == BASELINE_GROUP]
        for group, view in group_views(model_clips, NIGERIAN_GROUPS):
            gap = metrics.bootstrap_wer_difference(view["errors"], view["ref_words"],
                                                   base["errors"], base["ref_words"], seed=SEED)
            rows.append({"model": model, "group": group, "gap": gap.value,
                         "ci_low": gap.ci_low, "ci_high": gap.ci_high})
    return pd.DataFrame(rows)


def detail_tables(clips: pd.DataFrame, scores_by_model: dict) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Top substitutions per dataset, worst clips and every likely proper noun, for each model."""
    subs, worst, names = [], [], []
    group_of = dict(zip(clips["clip_id"], clips["group"]))
    for model, scores in scores_by_model.items():
        for data_label in dict.fromkeys(DATASET_OF_GROUP[g] for g in ALL_GROUPS):
            chosen = [s for cid, s in scores.items() if DATASET_OF_GROUP[group_of[cid]] == data_label]
            for rank, (ref, hyp, count) in enumerate(metrics.top_substitutions(chosen, n=25), start=1):
                subs.append({"model": model, "data": data_label, "rank": rank,
                             "reference_word": ref, "model_word": hyp, "count": count})
        for rank, (cid, score) in enumerate(metrics.worst_clips(scores.items(), n=30), start=1):
            row = clips[(clips["model"] == model) & (clips["clip_id"] == cid)].iloc[0]
            worst.append({"model": model, "rank": rank, "clip_id": cid, "group": row["group"],
                          "wer": score.counts.wer, "errors": score.counts.errors,
                          "reference": row["reference"], "hypothesis": row["hypothesis"],
                          "alignment": row["alignment"]})
        for cid, score in scores.items():
            for result in score.proper_nouns or ():
                names.append({"model": model, "clip_id": cid, "group": group_of[cid], "name": result.word,
                              "correct": result.correct, "heard_as": result.heard_as})
    return pd.DataFrame(subs), pd.DataFrame(worst), pd.DataFrame(names)

In [ ]:
clip_tables, scores_by_model = [], {}
for model_name in MODELS:
    preds = load_predictions(PRED_DIR / f"{short_name(model_name)}.csv")
    table, scores = score_predictions(manifest, preds, model_name)
    clip_tables.append(table)
    scores_by_model[short_name(model_name)] = scores
clip_scores = pd.concat(clip_tables, ignore_index=True)

summary = summarise(clip_scores, ALL_GROUPS)
gaps = gaps_vs_baseline(clip_scores)
top_subs, worst, proper_nouns = detail_tables(clip_scores, scores_by_model)

clip_scores.to_csv(RESULTS_DIR / "clip_scores.csv", index=False)
summary.to_csv(RESULTS_DIR / "summary_wer.csv", index=False)
gaps.to_csv(RESULTS_DIR / "gap_vs_librispeech.csv", index=False)
top_subs.to_csv(RESULTS_DIR / "top_substitutions.csv", index=False)
worst.to_csv(RESULTS_DIR / "worst_clips.csv", index=False)
proper_nouns.to_csv(RESULTS_DIR / "proper_noun_errors.csv", index=False)


def as_percent(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    shown = frame.copy()
    for column in columns:
        shown[column] = (shown[column] * 100).round(1)
    return shown


print("WER by group and model (%), corpus-level with 95% bootstrap CI")
display(as_percent(summary, ["wer", "ci_low", "ci_high", "sub_rate", "del_rate", "ins_rate",
                             "name_error_rate", "name_ci_low", "name_ci_high"]))
print("Gap vs LibriSpeech (percentage points). A CI that excludes 0 means the gap is not resampling noise.")
display(as_percent(gaps, ["gap", "ci_low", "ci_high"]))

### 5b. Robustness checks

Two checks on the headline numbers. Both reuse the saved transcriptions, so they need no GPU.

1. **Whisper's own text normaliser.** Our main normaliser works word by word (lowercase, strip punctuation), which keeps word positions for the name analysis. But it counts pure formatting differences as errors: "thirty" vs "30", "colour" vs "color", "onto" vs "on to". The Whisper paper scores with a fuller normaliser that standardises numbers, British spellings and contractions. If the Nigerian-vs-LibriSpeech gap survives that normaliser, it is not a formatting artefact.
2. **Speaker-level confidence intervals.** The main CIs resample clips, which treats 300 clips as independent. SLR70 has only ~31 speakers, and one speaker's clips share a voice, so resampling **whole speakers** gives a more honest, wider interval. The point estimate does not change.

Results go to `results/robustness.csv`. `normaliser = "ours"` is the main metric; `"whisper"` is the check.

In [ ]:
def load_whisper_normalizer():
    """Whisper's English normaliser, with the spelling map shipped alongside the model."""
    from transformers.models.whisper.english_normalizer import EnglishTextNormalizer
    mapping = json.loads(Path(hf_hub_download("openai/whisper-small", "normalizer.json")).read_text(encoding="utf-8"))
    return EnglishTextNormalizer(mapping)


def robustness_table(clips: pd.DataFrame, manifest: pd.DataFrame, normalize) -> pd.DataFrame:
    """WER and gap vs LibriSpeech under both normalisers, with clip- and speaker-level 95% CIs."""
    clips = clips.copy()
    clips["speaker"] = clips["clip_id"].map(dict(zip(manifest["clip_id"], manifest["speaker"])))
    whisper_counts = [metrics.counts_with_normalizer(r.reference, r.hypothesis, normalize)
                      for r in clips.itertuples(index=False)]
    clips["w_errors"] = [c.errors for c in whisper_counts]
    clips["w_ref_words"] = [c.ref_words for c in whisper_counts]

    rows = []
    for model, model_clips in clips.groupby("model", sort=False):
        base = model_clips[model_clips["group"] == BASELINE_GROUP]
        for group, view in group_views(model_clips, ALL_GROUPS):
            for normaliser, err, words in [("ours", "errors", "ref_words"), ("whisper", "w_errors", "w_ref_words")]:
                by_clip = metrics.bootstrap_wer(view[err], view[words], seed=SEED)
                by_speaker = metrics.bootstrap_wer_by_speaker(view[err], view[words], view["speaker"], seed=SEED)
                row = {"model": model, "group": group, "normaliser": normaliser,
                       "clips": len(view), "speakers": view["speaker"].nunique(), "wer": by_clip.value,
                       "clip_ci_low": by_clip.ci_low, "clip_ci_high": by_clip.ci_high,
                       "speaker_ci_low": by_speaker.ci_low, "speaker_ci_high": by_speaker.ci_high,
                       "gap": np.nan, "gap_speaker_ci_low": np.nan, "gap_speaker_ci_high": np.nan}
                if group != BASELINE_GROUP and not base.empty:
                    gap = metrics.bootstrap_wer_difference_by_speaker(
                        view[err], view[words], view["speaker"], base[err], base[words], base["speaker"], seed=SEED)
                    row.update(gap=gap.value, gap_speaker_ci_low=gap.ci_low, gap_speaker_ci_high=gap.ci_high)
                rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
robustness = robustness_table(clip_scores, manifest, load_whisper_normalizer())
robustness.to_csv(RESULTS_DIR / "robustness.csv", index=False)
print("Robustness: WER and gap vs LibriSpeech (%), with clip- and speaker-level 95% CIs")
display(as_percent(robustness, ["wer", "clip_ci_low", "clip_ci_high", "speaker_ci_low", "speaker_ci_high",
                                "gap", "gap_speaker_ci_low", "gap_speaker_ci_high"]))

## 6. Chart: WER by group and model

Model size is an ordered variable (tiny < base < small), so the bars use one blue that darkens with size instead of three unrelated colours. Error bars are 95% bootstrap CIs. Dashed lines separate datasets.

In [ ]:
MODEL_COLORS = {"whisper-tiny": "#86b6ef", "whisper-base": "#2a78d6", "whisper-small": "#104281"}
SURFACE, INK, MUTED_INK, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"


def plot_wer_by_group(summary: pd.DataFrame, groups: list[str], path: Path) -> None:
    models = list(dict.fromkeys(summary["model"]))
    x = np.arange(len(groups))
    width = 0.8 / len(models)

    fig, ax = plt.subplots(figsize=(max(8.0, 1.7 * len(groups) + 1.5), 5.4), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for k, model in enumerate(models):
        rows = summary[summary["model"] == model].set_index("group").reindex(groups)
        wer = rows["wer"] * 100
        error = [(rows["wer"] - rows["ci_low"]) * 100, (rows["ci_high"] - rows["wer"]) * 100]
        ax.bar(x + (k - (len(models) - 1) / 2) * width, wer, width, label=model,
               color=MODEL_COLORS.get(model, "#2a78d6"), edgecolor=SURFACE, linewidth=2, zorder=2,
               yerr=error, error_kw={"ecolor": MUTED_INK, "elinewidth": 1, "capsize": 3, "zorder": 3})

    clips_per_group = summary.drop_duplicates("group").set_index("group")["clips"]
    labels = [f"{g.replace(' (SLR70)', chr(10) + '(SLR70)')}\n(n={int(clips_per_group.get(g, 0))})" for g in groups]
    ax.set_xticks(x, labels, color=INK)
    for i in range(len(groups) - 1):
        if DATASET_OF_GROUP[groups[i]] != DATASET_OF_GROUP[groups[i + 1]]:
            ax.axvline(i + 0.5, color=GRID, linewidth=1, linestyle="--", zorder=1)
    ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
    ax.set_ylabel("Word error rate", color=MUTED_INK)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.tick_params(colors=MUTED_INK, length=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    # Legend in its own row above the plot, so it can never cover a bar.
    ax.legend(frameon=False, labelcolor=INK, ncol=len(models), loc="lower left",
              bbox_to_anchor=(0, 1.01), handlelength=1.2, borderaxespad=0)
    ax.set_title("Whisper word error rate: Nigerian-accented vs standard English",
                 loc="left", color=INK, fontsize=13, pad=40)
    ax.text(0, 1.085, "Zero-shot, forced English. Corpus-level WER; error bars are 95% bootstrap CIs over clips.",
            transform=ax.transAxes, color=MUTED_INK, fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=200, facecolor=SURFACE)
    plt.show()


plot_wer_by_group(summary, ALL_GROUPS, RESULTS_DIR / "wer_by_group.png")

## 7. Failure examples for the README

Pick three real examples to show in the README: ideally one mangled Nigerian name, one hallucination (WER > 100%), and one typical accent-related substitution.

SLR70 sentences mention many Nigerian people and places ("Ikechi Uko is from Abia State Nigeria"), which makes it a good source for the name question. AfriSpeech's general-domain sentences come from news and encyclopedia-style text, so fewer of its proper nouns are Nigerian.

In [ ]:
largest = short_name(MODELS[-1])
nigerian_names = proper_nouns[(proper_nouns["model"] == largest) & proper_nouns["group"].isin(NIGERIAN_GROUPS)]
print("Most common likely proper nouns in the Nigerian references:")
print(nigerian_names["name"].value_counts().head(30).to_dict())

print(f"\nName errors by {largest} (Nigerian groups):")
display(nigerian_names[~nigerian_names["correct"]].head(15))

for data_label in dict.fromkeys(DATASET_OF_GROUP[g] for g in NIGERIAN_GROUPS):
    print(f"\nTop substitutions by {largest} on {data_label}:")
    display(top_subs[(top_subs["model"] == largest) & (top_subs["data"] == data_label)].head(15))

print(f"\nWorst clips for {largest}:")
for row in worst[worst["model"] == largest].head(5).itertuples():
    print(f"[{row.group}] WER {row.wer:.0%}\n  REF: {row.reference}\n  HYP: {row.hypothesis}\n  ALN: {row.alignment}\n")

## 8. (Optional) Your own recordings

1. In Google Drive, create `MyDrive/naija-asr-eval/my_recordings/`.
2. Put your ~20 phone recordings there (any format ffmpeg reads: `.m4a`, `.mp3`, `.wav`, `.ogg`...). Keep each one under 30 seconds.
3. Add `my_recordings.csv` with two columns, `filename` and `transcript`, saying **exactly** what you said, with normal capitalisation (for example: `rec01.m4a`, `My name is Sobande and I live in Ibadan.`). `my_recordings_template.csv` in the repo shows the format.

The audio is cached privately like the rest. Only the scores and transcripts go into `results/`. With 20 clips the confidence intervals will be wide: treat this as qualitative evidence about names, not a benchmark.

In [ ]:
MY_DIR = WORK_DIR / "my_recordings"
MY_GROUP = "My recordings"


def decode_with_ffmpeg(path: Path) -> np.ndarray:
    """Decode any audio format to 16 kHz mono float32 with ffmpeg (preinstalled on Colab)."""
    command = ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", str(path),
               "-f", "f32le", "-ac", "1", "-ar", str(SAMPLE_RATE), "-"]
    result = subprocess.run(command, capture_output=True, check=True)
    return np.frombuffer(result.stdout, dtype=np.float32)


def build_my_manifest() -> pd.DataFrame:
    table = pd.read_csv(MY_DIR / "my_recordings.csv", keep_default_na=False, dtype=str)
    if not {"filename", "transcript"} <= set(table.columns):
        raise KeyError(f"my_recordings.csv needs columns 'filename' and 'transcript', found {list(table.columns)}")
    rows = []
    for item in table.itertuples(index=False):
        path = MY_DIR / item.filename
        if not path.exists():
            print(f"  skipping {item.filename}: file not found")
            continue
        if not str(item.transcript).strip():
            print(f"  skipping {item.filename}: empty transcript")
            continue
        clip_id = f"mine-{Path(item.filename).stem}"
        if not cache_path(clip_id).exists():
            try:
                audio = decode_with_ffmpeg(path)
            except subprocess.CalledProcessError as error:
                print(f"  skipping {item.filename}: ffmpeg failed ({error.stderr.decode(errors='ignore').strip()})")
                continue
            if len(audio) / SAMPLE_RATE > MAX_DURATION_S:
                print(f"  warning: {item.filename} is over {MAX_DURATION_S:.0f} s; Whisper will only hear the start")
            save_audio(clip_id, audio)
        rows.append({"clip_id": clip_id, "dataset": "mine", "group": MY_GROUP,
                     "reference": item.transcript, "filename": item.filename})
    return pd.DataFrame(rows)


if (MY_DIR / "my_recordings.csv").exists():
    my_manifest = build_my_manifest()
    my_pred_dir = PRED_DIR / "my_recordings"
    my_pred_dir.mkdir(exist_ok=True)
    my_tables, my_scores = [], {}
    for model_name in MODELS:
        out_path = my_pred_dir / f"{short_name(model_name)}.csv"
        transcribe_all(model_name, my_manifest["clip_id"].tolist(), out_path)
        table, scores = score_predictions(my_manifest, load_predictions(out_path), model_name)
        my_tables.append(table)
        my_scores[short_name(model_name)] = scores
    my_clip_scores = pd.concat(my_tables, ignore_index=True)
    my_summary = summarise(my_clip_scores, [MY_GROUP])
    my_names = pd.DataFrame([
        {"model": model, "clip_id": cid, "name": r.word, "correct": r.correct, "heard_as": r.heard_as}
        for model, scores in my_scores.items() for cid, s in scores.items() for r in (s.proper_nouns or ())
    ])
    my_clip_scores.to_csv(RESULTS_DIR / "my_recordings_scores.csv", index=False)
    my_summary.to_csv(RESULTS_DIR / "my_recordings_summary.csv", index=False)
    my_names.to_csv(RESULTS_DIR / "my_recordings_names.csv", index=False)
    display(as_percent(my_summary, ["wer", "ci_low", "ci_high", "sub_rate", "del_rate", "ins_rate",
                                    "name_error_rate", "name_ci_low", "name_ci_high"]))
    display(my_names[~my_names["correct"]] if not my_names.empty else my_names)
else:
    print(f"No {MY_DIR / 'my_recordings.csv'} found: skipping this optional section.")

## 9. Data, licences and citations

| Data | Licence | Notes |
|---|---|---|
| SLR70, Crowdsourced high-quality Nigerian English speech | CC BY-SA 4.0 | Results CSVs that contain its transcripts are shared under the same licence |
| AfriSpeech-200 | CC BY-NC-SA 4.0 | Non-commercial; transcripts in results CSVs inherit this |
| LibriSpeech | CC BY 4.0 | |

- Google. *Crowdsourced high-quality Nigerian English speech data set.* OpenSLR resource SLR70. https://openslr.org/70/
- Butryna, A., Chu, S.-H. C., Demirsahin, I., Gutkin, A., Ha, L., He, F., Jansche, M., Johny, C., Katanova, A., Kjartansson, O., Li, C., Merkulova, T., Oo, Y. M., Pipatsrisawat, K., Rivera, C., Sarin, S., de Silva, P., Sodimana, K., Sproat, R., Wattanavekin, T., & Wibawa, J. A. E. (2019). Google Crowdsourced Speech Corpora and Related Open-Source Resources for Low-Resource Languages and Dialects: An Overview. *2019 UNESCO International Conference Language Technologies for All (LT4All)*, Paris. arXiv:2010.06778.
- Olatunji, T., et al. (2023). AfriSpeech-200: Pan-African Accented Speech Dataset for Clinical and General Domain ASR. *Transactions of the Association for Computational Linguistics*. arXiv:2310.00274.
- Panayotov, V., Chen, G., Povey, D., & Khudanpur, S. (2015). Librispeech: An ASR corpus based on public domain audio books. *ICASSP 2015*.
- Radford, A., Kim, J. W., Xu, T., Brockman, G., McLeavey, C., & Sutskever, I. (2023). Robust Speech Recognition via Large-Scale Weak Supervision. *ICML 2023*.

## 10. Package the results

This cell checks that `results/` contains no audio, then zips it for download. Unzip it into the repo's `results/` folder on your laptop and commit. Never commit `audio_cache/` or the SLR70 zips.

In [ ]:
AUDIO_SUFFIXES = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac", ".amr", ".3gp", ".webm", ".mp4", ".zip"}
leaked = [p for p in RESULTS_DIR.rglob("*") if p.suffix.lower() in AUDIO_SUFFIXES]
if leaked:
    raise RuntimeError(f"Audio or archives found inside results/, refusing to package: {leaked[:5]}")

archive = shutil.make_archive("/content/naija_asr_results" if IN_COLAB else "naija_asr_results", "zip", RESULTS_DIR)
print("Packaged", archive)
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(RESULTS_DIR)}  ({path.stat().st_size / 1024:.0f} KB)")
if IN_COLAB:
    files.download(archive)